In [1]:
from rdkit.Chem import AllChem 
from numpy import zeros
from rdkit import DataStructs
from rdkit.Chem import MolFromSmiles
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, root_mean_squared_error
from pandas import DataFrame, concat
from pickle import load
import matplotlib.pyplot as plt

In [2]:
def calc_morgan(mols):
    """ генерация молекулярных отпечатков по методу Моргана с радиусом 2 и длиной 2048
    """
    mfp_gen = AllChem.GetMorganGenerator(radius=2, ) 
    for_df = []
    for m in mols:
        arr = zeros((1,), dtype=int)
        DataStructs.ConvertToNumpyArray(mfp_gen.GetFingerprint(m), arr)
        for_df.append(arr)
    return DataFrame(for_df)

with open('../../data/ld50/min_ld50.pickle', 'rb') as file:
    potency = load(file)

In [3]:
molecules = [
    MolFromSmiles(mol) for mol in potency['SMILES']
]

[10:12:06] WARNING: not removing hydrogen atom without neighbors
[10:12:06] WARNING: not removing hydrogen atom without neighbors
[10:12:07] WARNING: not removing hydrogen atom without neighbors
[10:12:07] WARNING: not removing hydrogen atom without neighbors
[10:12:07] WARNING: not removing hydrogen atom without neighbors
[10:12:07] WARNING: not removing hydrogen atom without neighbors
[10:12:07] WARNING: not removing hydrogen atom without neighbors
[10:12:07] WARNING: not removing hydrogen atom without neighbors
[10:12:07] WARNING: not removing hydrogen atom without neighbors
[10:12:07] WARNING: not removing hydrogen atom without neighbors
[10:12:07] WARNING: not removing hydrogen atom without neighbors
[10:12:07] WARNING: not removing hydrogen atom without neighbors
[10:12:07] WARNING: not removing hydrogen atom without neighbors


In [4]:
X = calc_morgan(molecules)
Y = potency['LD50']
XY = concat((X, Y), axis=1)
XY

,0,1,2,3,4,5,6,7,8,9,...,2039,2040,2041,2042,2043,2044,2045,2046,2047,LD50
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,25.0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1930.0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,8471.0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1460.0
4,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,640.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23504,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1470.0
23505,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,470.0
23506,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,325.0
23507,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1300.0


In [5]:
scaler = StandardScaler().fit(Y.to_numpy().reshape(-1, 1))
Y = scaler.transform(Y.to_numpy().reshape(-1, 1))

In [6]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.20) # попробовать 20, 15

In [7]:
gs = RandomForestRegressor().fit(x_train, y_train.ravel())

In [8]:
y_pred = gs.predict(x_test)
print(f'R^2 = {r2_score(y_test, y_pred)}')
print(f'RMSE = {root_mean_squared_error(y_test, y_pred)}')

R^2 = -17.45021893858638
RMSE = 0.7785353575228234
